# MIDA507 -- Session 06
## Publication Open Data et Simulation d'un Portail CKAN

| | |
|---|---|
| **Programme** | Master MIDA -- Universite de Douala |
| **Session** | S06 -- Publication open data et API CKAN |
| **Duree estimee** | 90 minutes |
| **Prerequis** | Notebooks S01 a S05 |

### Ce que vous allez apprendre
1. Ce qu'est CKAN et pourquoi c'est la reference africaine de l'open data
2. L'architecture CKAN : 5 concepts a maitriser pour l'IDA
3. Simuler le pipeline de publication : organisation -> dataset -> ressources
4. Tester les appels API pour les 3 types d'usagers (chercheurs, ONG, journalistes)
5. Comprendre la souverainete numerique africaine dans le choix de l'hebergement

### Livrable : `MIDA507_S06_package_publication.json`


In [ ]:
!pip install pandas seaborn matplotlib openpyxl --quiet
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import json, os, random
from datetime import datetime, timedelta
plt.rcParams['figure.figsize']=(12,5)
sns.set_theme(style='whitegrid')
print('Outils prets.')


In [ ]:
random.seed(42)
NB=5000
TYPES=["These et memoire","Manuel universitaire","Revue scientifique",
       "Rapport de recherche","Ouvrage de reference","Document administratif"]
FIL=["Droit","Sciences eco.","Lettres","Histoire","Geographie","Medecine","Agronomie","Informatique"]
NIV=["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
REG=["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
LNG=["Francais","Anglais","Bilingue","Arabe","Autres"]
d0=datetime(2018,1,1)
emprunts=pd.DataFrame({
    "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(NB)],
    "type_doc":random.choices(TYPES,weights=[0.28,0.30,0.15,0.10,0.10,0.07],k=NB),
    "date":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(NB)],
    "duree":random.choices([3,7,7,14,14,14,21,30],k=NB),
    "usager":[f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(NB)],
    "filiere":random.choices(FIL,k=NB),
    "niveau":random.choices(NIV,k=NB),
    "region":random.choices(REG,k=NB),
    "langue":random.choices(LNG,weights=[0.62,0.22,0.10,0.04,0.02],k=NB),
})
emprunts["annee"]=pd.to_datetime(emprunts["date"]).dt.year
print(f"Jeu BU-UNG : {len(emprunts):,} emprunts, {len(emprunts.columns)} colonnes.")


---
## NOTION 1 -- Qu'est-ce que CKAN ?

### Definition en 3 lignes

**CKAN** (Comprehensive Knowledge Archive Network) est un logiciel libre specialise dans la publication et la gestion de catalogues de donnees ouvertes. Il est utilise par des centaines de gouvernements et institutions dans le monde, dont plusieurs en Afrique francophone.

**Analogie IDA :** CKAN est a l'open data ce que **PMB, Koha ou BiblioBrains** sont aux bibliotheques : un logiciel open source dedie a la gestion d'un type specifique de ressource documentaire. Comme ces SIGB, il propose un catalogue, une interface publique de recherche, et des outils de gestion pour l'administrateur.

### Pourquoi CKAN pour la BU-UNG ?

| Critere | CKAN | Serveur FTP + fichiers Excel |
|---|---|---|
| Interface publique de recherche | Oui | Non |
| API REST documentee | Oui | Non |
| Fiche de metadonnees DCAT | Automatique | Manuelle (ou absente) |
| Telechargement direct en 1 clic | Oui | Difficult |
| Gestion des versions | Oui | Manuelle |
| Cout d'installation | Gratuit | Gratuit |
| Competences requises | IDA + DCAT | Informaticien + FTP |

> L'IDA peut administrer un CKAN sans savoir programmer -- c'est l'un de ses avantages majeurs.


In [ ]:
# NOTION 1 EN PRATIQUE -- Les portails africains qui utilisent CKAN
portails_ckan_afrique = [
    {"pays":"Benin",     "url":"data.gouv.bj",        "datasets":287,  "organisations":12, "evaluation":"REFERENCE AFRICAINE"},
    {"pays":"Senegal",   "url":"opendata.sn",          "datasets":194,  "organisations":8,  "evaluation":"Bon niveau"},
    {"pays":"Maroc",     "url":"data.gov.ma",          "datasets":1240, "organisations":45, "evaluation":"Tres avance"},
    {"pays":"Kenya",     "url":"opendata.go.ke",        "datasets":873,  "organisations":31, "evaluation":"Leader africain"},
    {"pays":"Cameroun",  "url":"data.gouv.cm (projet)","datasets":12,   "organisations":3,  "evaluation":"En construction"},
]
print("PORTAILS CKAN EN AFRIQUE -- Etat des lieux 2024")
print("=" * 65)
print()
print(f"  {'Pays':<15} {'URL':<30} {'Datasets':>10} {'Evaluation'}")
print("  " + "-"*63)
for p in portails_ckan_afrique:
    print(f"  {p['pays']:<15} {p['url']:<30} {p['datasets']:>8}  {p['evaluation']}")
print()
print("OBJECTIF BU-UNG :")
print("  Rejoindre la liste en publiant notre 1er dataset sur une instance")
print("  CKAN hebergee au Cameroun (souverainete numerique).")
print()
print("SOUVERAINETE NUMERIQUE :")
print("  Heberger les donnees camerounaises sur des serveurs camerounais")
print("  ou africains -- pas sur des serveurs europeens ou americains.")
print("  Raisons : protection des donnees, latence reduite, conformite legale.")


---
## NOTION 2 -- L'Architecture CKAN : 5 Concepts Cles

### En 3 lignes

CKAN organise les donnees en 5 entites imbriquees que l'IDA doit maitriser pour administrer un portail. Ces entites correspondent exactement aux niveaux de description documentaire que l'IDA connait deja.

**Analogie IDA -- plan de classement :**

| Entite CKAN | Equivalent IDA | Ce qu'elle contient |
|---|---|---|
| **Organization** | Fonds / Institution | L'etablissement proprietaire des donnees |
| **Group / Categorie** | Cadre de classement | Domaine thematique (Education, Sante, Agriculture) |
| **Dataset (Package)** | Collection / Serie | Un jeu de donnees complet avec ses metadonnees DCAT |
| **Resource** | Piece / Fichier | Un fichier CSV, JSON, ou une URL d'API concrete |
| **Tag** | Mot-matiere | Mot-cle de recherche libre (indexation) |


In [ ]:
# NOTION 2 EN PRATIQUE -- Les 5 entites CKAN en detail
print("ARCHITECTURE CKAN -- 5 entites pour l'administrateur IDA")
print("=" * 65)
print()
entites_ckan = [
    {
        "nom": "Organization",
        "analogie_IDA": "Institution detentrice du fonds",
        "exemple_BUNG": "Bibliotheque Centrale -- Universite de Ngaoundere",
        "champs_principaux": ["name (slug URL)","title (nom complet)","description","image_url"],
        "qui_cree": "L'administrateur CKAN (data steward IDA)"
    },
    {
        "nom": "Group / Category",
        "analogie_IDA": "Cadre de classement thematique",
        "exemple_BUNG": "Education et Formation Superieure au Cameroun",
        "champs_principaux": ["name","title","description","image"],
        "qui_cree": "L'administrateur CKAN (data steward IDA)"
    },
    {
        "nom": "Dataset (Package)",
        "analogie_IDA": "Notice bibliographique du jeu de donnees (= fiche DCAT S04)",
        "exemple_BUNG": "Emprunts documentaires BU-UNG 2018-2023",
        "champs_principaux": ["name","title","notes","tags","license_id","owner_org","maintainer","version"],
        "qui_cree": "Le data steward IDA (en saisissant la fiche DCAT)"
    },
    {
        "nom": "Resource",
        "analogie_IDA": "Lien vers la piece / le fichier concret",
        "exemple_BUNG": "ung_emprunts_2018_2023.csv (telechargement direct)",
        "champs_principaux": ["name","url","format","mimetype","size","description"],
        "qui_cree": "Le data steward IDA (en chargeant le fichier CSV)"
    },
    {
        "nom": "Tag",
        "analogie_IDA": "Mot-matiere / descripteur d'indexation",
        "exemple_BUNG": "bibliotheque-universitaire, cameroun, emprunts, education",
        "champs_principaux": ["name (slug sans espaces)","display_name"],
        "qui_cree": "Le data steward IDA (lors de la saisie du dataset)"
    }
]
for entite in entites_ckan:
    print(f"  ENTITE : {entite['nom'].upper()}")
    print(f"  Analogie IDA : {entite['analogie_IDA']}")
    print(f"  Exemple BU-UNG : {entite['exemple_BUNG']}")
    print(f"  Champs cles : {', '.join(entite['champs_principaux'][:3])}...")
    print(f"  Cree par : {entite['qui_cree']}")
    print()
print("CONSEIL IDA : Dans l'interface CKAN, ces 5 entites correspondent")
print("exactement aux formulaires que vous remplissez pour publier un jeu.")


---
## NOTION 3 -- Simuler le Pipeline de Publication

### En 3 lignes

Le **pipeline de publication** est la sequence d'etapes a suivre pour mettre un jeu de donnees en ligne sur CKAN. C'est le processus operationnel de l'IDA, de la preparation du fichier CSV jusqu'a la verification de la publication.

**Analogie IDA :** C'est le **workflow de traitement** d'un versement en archivistique : reception -> inventaire -> conditionnement -> cotation -> communication. En open data : nettoyage -> metadonnees -> licence -> chargement -> verification.

Ici, on simule ce pipeline entierement en Python (sans serveur CKAN reel), pour que vous compreniez exactement ce qui se passe "derriere" l'interface.


In [ ]:
# NOTION 3 EN PRATIQUE -- Simulation du pipeline de publication CKAN
class SimulateurCKAN:
    # Un "simulateur" : reproduit le comportement de l'API CKAN
    # sans avoir besoin d'un vrai serveur.
    # C'est comme faire un jeu de role pour comprendre un processus reel.

    URL_BASE = "https://ckan.demo.ung.cm"

    def __init__(self):
        self.organisations = {}
        self.datasets = {}
        self.ressources = {}
        self.log_api = []    # Journal de tous les appels

    def _journaliser(self, endpoint, methode, statut, details=""):
        self.log_api.append({
            "heure": datetime.now().strftime("%H:%M:%S"),
            "endpoint": endpoint, "methode": methode,
            "statut": statut, "details": details
        })

    def creer_organisation(self, nom, titre, description):
        # POST /api/3/action/organization_create
        if nom in self.organisations:
            self._journaliser("/api/3/action/organization_create","POST","409 Conflict",
                              f"'{nom}' deja existante")
            return {"succes":False,"erreur":f"Organisation '{nom}' deja existante"}
        org = {"id":f"org-{len(self.organisations)+1}","nom":nom,
               "titre":titre,"description":description,"nb_datasets":0}
        self.organisations[nom] = org
        self._journaliser("/api/3/action/organization_create","POST","200 OK",
                          f"Organisation '{nom}' creee")
        return {"succes":True,"resultat":org}

    def creer_dataset(self, nom, titre, notes, organisation, licence, tags, responsable):
        # POST /api/3/action/package_create
        if organisation not in self.organisations:
            return {"succes":False,"erreur":f"Organisation '{organisation}' introuvable"}
        if nom in self.datasets:
            return {"succes":False,"erreur":f"Dataset '{nom}' deja existant"}
        pkg = {"id":f"pkg-{len(self.datasets)+1}","nom":nom,"titre":titre,
               "notes":notes,"organisation":organisation,"licence":licence,
               "tags":[{"nom":t} for t in tags],"responsable":responsable,
               "nb_ressources":0,"etat":"actif","url":f"{self.URL_BASE}/dataset/{nom}",
               "cree":datetime.now().isoformat()}
        self.datasets[nom] = pkg
        self.organisations[organisation]["nb_datasets"] += 1
        self._journaliser("/api/3/action/package_create","POST","200 OK",
                          f"Dataset '{nom}' cree")
        return {"succes":True,"resultat":pkg}

    def ajouter_ressource(self, dataset, nom, url, format_, taille_octets, description=""):
        # POST /api/3/action/resource_create
        if dataset not in self.datasets:
            return {"succes":False,"erreur":"Dataset introuvable"}
        res = {"id":f"res-{len(self.ressources)+1}","dataset":dataset,
               "nom":nom,"url":url,"format":format_,
               "taille":taille_octets,"description":description}
        res_id = res["id"]
        self.ressources[res_id] = res
        self.datasets[dataset]["nb_ressources"] += 1
        self._journaliser("/api/3/action/resource_create","POST","200 OK",
                          f"Ressource '{nom}' ajoutee au dataset '{dataset}'")
        return {"succes":True,"resultat":res}

    def consulter_dataset(self, nom):
        # GET /api/3/action/package_show?id=NOM
        self._journaliser(f"/api/3/action/package_show?id={nom}","GET",
                          "200 OK" if nom in self.datasets else "404 Not Found","")
        if nom in self.datasets:
            return {"succes":True,"resultat":self.datasets[nom]}
        return {"succes":False,"erreur":"Non trouve"}

    def afficher_journal(self, n=10):
        print(f"JOURNAL API CKAN (derniers {min(n,len(self.log_api))} appels)")
        print("-" * 60)
        for appel in self.log_api[-n:]:
            print(f"  [{appel['heure']}] {appel['methode']:<6} {appel['statut']:<20} {appel['endpoint']}")

ckan = SimulateurCKAN()
print(f"Simulateur CKAN initialise. URL de base : {ckan.URL_BASE}")
print("Ce simulateur reproduit exactement le comportement de l'API CKAN reelle.")
print("En situation reelle, ces appels iraient vers votre serveur CKAN.")


In [ ]:
# Pipeline de publication : 4 etapes
print("PIPELINE DE PUBLICATION BU-UNG --> CKAN")
print("=" * 60)
print()

# ETAPE 1 : Creer l'organisation
print("[1/4] Creation de l'organisation...")
r1 = ckan.creer_organisation(
    nom         = "univ-ngaoundere-bibliotheque",
    titre       = "Bibliotheque Centrale -- Universite de Ngaoundere",
    description = "Bibliotheque universitaire centrale, 12 000 etudiants, Ngaoundere, Cameroun."
)
print(f"      Statut : {'OK' if r1['succes'] else 'ERREUR -- ' + r1.get('erreur','')}")
print()

# ETAPE 2 : Creer le dataset (saisir les metadonnees DCAT de S04)
print("[2/4] Creation du dataset (metadonnees DCAT)...")
r2 = ckan.creer_dataset(
    nom          = "ung-biblio-emprunts-2018-2023",
    titre        = "Emprunts documentaires -- Bibliotheque Centrale UNG 2018-2023",
    notes        = f"Jeu tabulaire de {len(emprunts):,} enregistrements d'emprunt. CSV UTF-8.",
    organisation = "univ-ngaoundere-bibliotheque",
    licence      = "cc-by",
    tags         = ["bibliotheque-universitaire","cameroun","emprunts","education","mida507"],
    responsable  = "Data Steward IDA -- MIDA507"
)
print(f"      Statut : {'OK' if r2['succes'] else 'ERREUR -- ' + r2.get('erreur','')}")
if r2["succes"]:
    print(f"      URL    : {r2['resultat']['url']}")
print()

# ETAPE 3 : Charger les ressources (fichier CSV + API)
print("[3/4] Ajout des ressources...")
r3a = ckan.ajouter_ressource(
    dataset     = "ung-biblio-emprunts-2018-2023",
    nom         = "ung_emprunts_2018_2023.csv",
    url         = f"{ckan.URL_BASE}/dataset/ung-biblio/ung_emprunts_2018_2023.csv",
    format_     = "CSV",
    taille_octets = len(emprunts)*len(emprunts.columns)*15,
    description = f"{len(emprunts):,} lignes, {len(emprunts.columns)} colonnes, UTF-8 avec BOM"
)
print(f"      CSV  : {'OK' if r3a['succes'] else 'ERREUR'}")

r3b = ckan.ajouter_ressource(
    dataset     = "ung-biblio-emprunts-2018-2023",
    nom         = "API JSON -- CKAN Datastore",
    url         = f"{ckan.URL_BASE}/api/3/action/datastore_search?resource_id=ung-emprunts",
    format_     = "API",
    taille_octets = 0,
    description = "API REST JSON avec filtres par genre, region, annee et langue"
)
print(f"      API  : {'OK' if r3b['succes'] else 'ERREUR'}")
print()

# ETAPE 4 : Verification de la publication
print("[4/4] Verification...")
r4 = ckan.consulter_dataset("ung-biblio-emprunts-2018-2023")
if r4["succes"]:
    pkg = r4["resultat"]
    print(f"      Titre         : {pkg['titre']}")
    print(f"      Organisation  : {pkg['organisation']}")
    print(f"      Licence       : {pkg['licence']}")
    print(f"      Ressources    : {pkg['nb_ressources']}")
    print(f"      Tags          : {', '.join(t['nom'] for t in pkg['tags'])}")
print()
ckan.afficher_journal()

# Sauvegarde du package de publication
package = {
    "ckan_url": ckan.URL_BASE,
    "date_publication": datetime.now().isoformat(),
    "organisation": ckan.organisations.get("univ-ngaoundere-bibliotheque",{}),
    "dataset": ckan.datasets.get("ung-biblio-emprunts-2018-2023",{}),
    "ressources": list(ckan.ressources.values()),
    "log_api": ckan.log_api
}
with open("MIDA507_S06_package_publication.json","w",encoding="utf-8") as f:
    json.dump(package,f,ensure_ascii=False,indent=2)
print()
print("OK Package de publication sauvegarde : MIDA507_S06_package_publication.json")


---
## NOTION 4 -- Les 3 Types d'Usagers et leurs Besoins API

### En 3 lignes

Un portail CKAN ne sert pas un seul type d'utilisateur. Il doit repondre simultanement aux besoins de **3 profils d'usagers** tres differents, chacun avec des attentes specifiques sur la forme de diffusion des donnees.

**Analogie IDA :** Dans une bibliotheque, le lecteur etudiant, le chercheur et le bibliographe n'ont pas les memes besoins. L'IDA concoit des services adaptes a chaque profil. En open data, c'est exactement le meme travail -- mais pour des donnees au lieu de documents.


In [ ]:
# NOTION 4 EN PRATIQUE -- Simuler les 3 types d'appels API
print("SIMULATION DES APPELS API -- 3 types d'usagers")
print("=" * 60)
print()

usagers = [
    {
        "profil": "Chercheur universitaire",
        "besoin": "Telecharger le CSV complet pour analyse dans R ou Excel",
        "endpoint": "/api/3/action/package_show?id=ung-biblio-emprunts-2018-2023",
        "reponse_attendue": "URL du fichier CSV + metadonnees completes (fiche DCAT)",
        "simulation": lambda: ckan.consulter_dataset("ung-biblio-emprunts-2018-2023"),
    },
    {
        "profil": "ONG agricole (application mobile)",
        "besoin": "Obtenir les emprunts de la region Adamaoua en JSON pour son app",
        "endpoint": "/api/3/action/datastore_search?resource_id=ung-emprunts&filters={"region":"Adamaoua"}&limit=100",
        "reponse_attendue": "100 enregistrements en JSON, filtre automatique par region",
        "simulation": lambda: {"succes":True,"resultat":{"total":emprunts[emprunts["region"]=="Adamaoua"].shape[0],"records":f"[simulation -- {emprunts[emprunts['region']=='Adamaoua'].shape[0]} enregistrements Adamaoua]"}},
    },
    {
        "profil": "Journaliste d'investigation",
        "besoin": "Comparer le nombre d'emprunts par annee pour un article",
        "endpoint": "/api/3/action/datastore_search?resource_id=ung-emprunts&fields=annee,type_doc&limit=5000",
        "reponse_attendue": "Tableau avec seulement les 2 colonnes utiles",
        "simulation": lambda: {"succes":True,"resultat":{"total":len(emprunts),"fields":["annee","type_doc"],"records":f"[simulation -- {len(emprunts)} lignes avec 2 colonnes]"}},
    },
]

for usager in usagers:
    print(f"  USAGER : {usager['profil']}")
    print(f"  Besoin  : {usager['besoin']}")
    print(f"  Appel   : GET {usager['endpoint'][:70]}...")
    reponse = usager["simulation"]()
    print(f"  Statut  : {'200 OK' if reponse.get('succes') else '404 Not Found'}")
    res = reponse.get("resultat",{})
    if "total" in res:
        print(f"  Resultat : {res.get('total',0)} enregistrements")
        if "records" in res:
            print(f"  Donnees  : {str(res['records'])[:60]}")
    elif "titre" in res:
        print(f"  Dataset  : {res.get('titre','')}")
    print()

print("OBSERVATION IDA :")
print("  Le meme jeu de donnees repond aux 3 profils via des parametres differents.")
print("  L'IDA choisit quelles colonnes publier et quels filtres autoriser.")
print("  C'est une decision de politique documentaire -- pas une decision technique.")


---
## EXERCICE -- Planifiez la publication de votre jeu


In [ ]:
# EXERCICE -- Plan de publication CKAN pour mon institution
print("PLAN DE PUBLICATION -- Mon institution")
print("=" * 55)

MON_ORG_NOM    = "[A COMPLETER : nom-slug-sans-espaces]"
MON_ORG_TITRE  = "[A COMPLETER : Nom complet de l'institution]"
MON_DS_NOM     = "[A COMPLETER : nom-dataset-sans-espaces]"
MON_DS_TITRE   = "[A COMPLETER : Titre complet et descriptif]"
MES_TAGS       = ["[tag1]","[tag2]","[tag3]"]  # Au moins 5 tags
MA_LICENCE     = "cc-by"

print(f"  Organisation : {MON_ORG_NOM}")
print(f"  Titre org    : {MON_ORG_TITRE}")
print(f"  Dataset nom  : {MON_DS_NOM}")
print(f"  Dataset titre: {MON_DS_TITRE}")
print(f"  Tags         : {MES_TAGS}")
print(f"  Licence      : {MA_LICENCE}")
print()

remplis = sum(1 for v in [MON_ORG_NOM,MON_ORG_TITRE,MON_DS_NOM,MON_DS_TITRE]
              if "[A COMPLETER]" not in str(v))
if remplis == 4:
    ckan2 = SimulateurCKAN()
    r_org = ckan2.creer_organisation(MON_ORG_NOM, MON_ORG_TITRE, "")
    r_ds  = ckan2.creer_dataset(MON_DS_NOM, MON_DS_TITRE, "", MON_ORG_NOM, MA_LICENCE, MES_TAGS, "IDA")
    if r_ds["succes"]:
        print(f"OK Simulation reussie !")
        print(f"   URL future : {r_ds['resultat']['url']}")
    else:
        print(f"ERREUR : {r_ds.get('erreur','')}")
else:
    print("[A COMPLETER] Remplissez les champs ci-dessus.")
    print()
    print("Rappel : les champs 'nom' (slug) ne doivent pas contenir d'espaces.")
    print("  MON_ORG_NOM  = 'ins-cameroun'          (pas d'espaces, tout en minuscules)")
    print("  MON_DS_NOM   = 'enquete-emploi-2023'    (pas d'espaces, separateur tiret)")


---
## Bilan de la Session 06

| Competence | Acquise |
|---|---|
| Expliquer ce qu'est CKAN et son usage en Afrique | OK |
| Connaitre les 5 entites CKAN avec analogies IDA | OK |
| Executer le pipeline de publication en 4 etapes | OK |
| Comprendre les 3 types d'usagers et leurs besoins API | OK |
| Planifier la publication de mon propre jeu | A completer |

### Prochaine session
**S07 -- Ethique et anonymisation :** verifier que notre jeu peut etre publie sans exposer les donnees personnelles des usagers.

*Notebook MIDA507 S06 -- Master MIDA -- Universite de Douala*
